# Robustness check: does the sector result survive without NVDA?

NVDA rose by roughly 6,500% over the window, so it is fair to ask whether one extreme stock is driving the finding that Technology is more volatile than the other sectors. This notebook reruns the sector volatility test without NVDA, then drops each stock in turn to see how fragile the result is.

**Method:** same as notebook 04. One annualised volatility value per stock, Kruskal-Wallis across sectors, pairwise exact Mann-Whitney U tests, Holm correction for three comparisons.

In [9]:
import itertools
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv('../data/processed/risk_metrics.csv')
VOL, SECTOR = 'ann_vol', 'sector'

def holm(pvals):
    """Holm-adjusted p-values for a dict {name: p}."""
    ordered = sorted(pvals.items(), key=lambda kv: kv[1])
    m, running, out = len(ordered), 0.0, {}
    for i, (k, p) in enumerate(ordered):
        running = max(running, min(1.0, (m - i) * p))
        out[k] = running
    return out

def sector_test(data):
    groups = {s: g[VOL].values for s, g in data.groupby(SECTOR)}
    kw = stats.kruskal(*groups.values()).pvalue
    raw = {(a, b): stats.mannwhitneyu(groups[a], groups[b], method='exact').pvalue
           for a, b in itertools.combinations(sorted(groups), 2)}
    return kw, raw, holm(raw), data.groupby(SECTOR)[VOL].median()

## 1. Full basket versus without NVDA

In [10]:
rows = []
for label, data in [('All 18 stocks', df), ('Without NVDA (17)', df[df.ticker != 'NVDA'])]:
    kw, raw, adj, med = sector_test(data)
    rows.append({'sample': label,
                 'Kruskal-Wallis p': round(kw, 3),
                 'Tech vs FS, raw p': round(raw[('Financial Services', 'Technology')], 3),
                 'Tech vs FS, Holm p': round(adj[('Financial Services', 'Technology')], 3),
                 'Median vol Tech %': round(med['Technology'], 1),
                 'Median vol Healthcare %': round(med['Healthcare'], 1),
                 'Median vol FS %': round(med['Financial Services'], 1)})
pd.DataFrame(rows).set_index('sample').T

sample,All 18 stocks,Without NVDA (17)
Kruskal-Wallis p,0.039,0.065
"Tech vs FS, raw p",0.009,0.017
"Tech vs FS, Holm p",0.026,0.052
Median vol Tech %,55.000,56.000
Median vol Healthcare %,37.200,37.200
Median vol FS %,34.800,34.800


## 2. Is NVDA actually a volatility outlier?

In [11]:
df[df.sector == 'Technology'][['ticker', VOL]].sort_values(VOL)

,ticker,ann_vol
13,CRM,38.86
0,NVDA,50.46
2,MU,54.02
3,AMD,55.99
7,DDOG,60.19
15,SNOW,62.11


NVDA's volatility (about 50%) is the second-lowest in Technology. It is a **return** outlier, not a **volatility** outlier, and this test uses volatility. Removing it therefore does not remove an extreme value; it removes one of six Technology observations, which cuts statistical power.

## 3. Leave-one-out: drop each stock in turn

In [12]:
rows = []
for t in df.ticker:
    kw, raw, adj, _ = sector_test(df[df.ticker != t])
    rows.append({'dropped': t, 'KW p': kw,
                 'Tech vs FS Holm p': adj[('Financial Services', 'Technology')]})
loo = pd.DataFrame(rows).round(3)
print(loo.sort_values('KW p').to_string(index=False))
print()
print('Runs with Kruskal-Wallis p < 0.05:', (loo['KW p'] < 0.05).sum(), 'of', len(loo))
print('Runs with Tech vs FS Holm p < 0.05:', (loo['Tech vs FS Holm p'] < 0.05).sum(), 'of', len(loo))

dropped  KW p  Tech vs FS Holm p
   ALHC 0.014              0.026
    CRM 0.024              0.013
    NVS 0.032              0.026
     BX 0.038              0.026
    KKR 0.038              0.026
    UNH 0.041              0.026
    LLY 0.041              0.026
   BIIB 0.049              0.026
    MOH 0.049              0.026
    AXP 0.051              0.052
    WFC 0.051              0.052
    JPM 0.060              0.052
      V 0.060              0.052
    AMD 0.065              0.052
   DDOG 0.065              0.052
   SNOW 0.065              0.052
     MU 0.065              0.052
   NVDA 0.065              0.052

Runs with Kruskal-Wallis p < 0.05: 9 of 18
Runs with Tech vs FS Holm p < 0.05: 9 of 18


## Interpretation

- Without NVDA the Kruskal-Wallis p-value moves from 0.039 to 0.065, and the Technology vs Financial Services Holm-adjusted p-value from 0.026 to 0.052. Both cross the 5% line.
- NVDA is not inflating the result: the Technology median volatility barely changes. The effect is lost because six stocks per sector leaves little statistical power.
- The leave-one-out table shows how dependent the finding is on individual stocks, so I describe the result as **suggestive, not confirmed**.
- **Next step:** a larger, randomly selected sample of stocks per sector, which would remove both the power problem and the hand-picking bias.